In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, f1_score, classification_report,
    average_precision_score, hamming_loss
)
from sklearn.calibration import CalibratedClassifierCV

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import joblib
import time

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[INFO] Using device: {DEVICE}')
print(f'[INFO] PyTorch Version: {torch.__version__}')
print(f'[INFO] CUDA Available: {torch.cuda.is_available()}')

In [ ]:
# ── Paths ────────────────────────────────────────────────────
BASE_DIR = r'C:\\Users\\gamer\\OneDrive\\Desktop\\IIT Project\\DataProcessing'

TRAIN_FEATURES_PATH = os.path.join(BASE_DIR, 'extracted_features_train.npy')
TEST_FEATURES_PATH  = os.path.join(BASE_DIR, 'extracted_features_test.npy')
TRAIN_CSV_PATH      = os.path.join(BASE_DIR, 'train_final.csv')
TEST_CSV_PATH       = os.path.join(BASE_DIR, 'test_final.csv')
MODEL_SAVE_DIR      = os.path.join(BASE_DIR, 'mce_models')
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# ── 14 Target Pathologies ────────────────────────────────────
PATHOLOGIES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration',
    'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax',
    'Consolidation', 'Edema', 'Emphysema', 'Fibrosis',
    'Pleural_Thickening', 'Hernia'
]
NUM_PATHOLOGIES = len(PATHOLOGIES)  # 14

# ── Clinical Metadata Columns ────────────────────────────────
META_COLS = ['ViewPosition_encoded', 'Age_standardized', 'Gender_encoded']

# ── Positivity Weight Vector (Clinical Calibration) ──────────
POS_WEIGHTS = torch.tensor(
    [0.64, 1.04, 0.50, 0.37, 1.29, 1.26,
     1.94, 1.35, 1.54, 1.21, 1.13, 1.65, 2.21, 11.73],
    dtype=torch.float32
).to(DEVICE)

# ── Hyperparameters ──────────────────────────────────────────
CONFIG = {
    'VISUAL_FEATURE_DIM' : 1024,
    'META_FEATURE_DIM'   : 3,
    'AUGMENTED_DIM'      : 1027,   # 1024 + 3
    'BATCH_SIZE'         : 256,
    'LEARNING_RATE'      : 1e-3,
    'EPOCHS'             : 50,
    'DROPOUT_RATE'       : 0.5,
    'EARLY_STOP_PATIENCE': 7,
    'RF_N_ESTIMATORS'    : 500,
    'SVM_C'              : 1.0,
    'SVM_KERNEL'         : 'rbf',
    'META_TEST_SIZE'     : 0.15,
}

print('[CONFIG] Phase 4 – MCE Configuration Loaded')
print(f'         Visual Feature Dim  : {CONFIG["VISUAL_FEATURE_DIM"]}')
print(f'         Augmented Input Dim : {CONFIG["AUGMENTED_DIM"]}')
print(f'         Num Pathologies     : {NUM_PATHOLOGIES}')
for i, (p, w) in enumerate(zip(PATHOLOGIES, POS_WEIGHTS.cpu().tolist())):
    print(f'         [{i+1:2d}] {p:<22s}  pos_weight={w:.2f}')

In [ ]:
def load_and_augment(features_path, csv_path, split_name='Train'):
    print(f'\\n[DATA] Loading {split_name} split...')

    # ── Load visual features ─────────────────────────────────
    X_visual = np.load(features_path, allow_pickle=True).astype(np.float32)
    print(f'       Visual features shape : {X_visual.shape}')
    assert X_visual.shape[1] == CONFIG['VISUAL_FEATURE_DIM'], \
        f'Expected 1024-dim features, got {X_visual.shape[1]}'

    # ── Load CSV ──────────────────────────────────────────────
    df = pd.read_csv(csv_path)
    print(f'       CSV shape             : {df.shape}')
    print(f'       CSV columns           : {list(df.columns)}')
    assert len(df) == len(X_visual), \
        f'Row mismatch: features={len(X_visual)}, csv={len(df)}'

    # ── Extract & validate metadata ───────────────────────────
    for col in META_COLS:
        if col not in df.columns:
            raise KeyError(f'Required metadata column "{col}" not found in CSV.'
                           f' Available: {list(df.columns)}')
    X_meta = df[META_COLS].values.astype(np.float32)
    print(f'       Metadata shape        : {X_meta.shape}')
    print(f'       ViewPosition  unique  : {df[META_COLS[0]].unique()}')
    print(f'       Gender        unique  : {df[META_COLS[2]].unique()}')
    print(f'       Age (std)     range   : [{df[META_COLS[1]].min():.3f}, {df[META_COLS[1]].max():.3f}]')

    # ── Concatenate → 1,027-dim augmented vector ──────────────
    X_aug = np.concatenate([X_visual, X_meta], axis=1).astype(np.float32)
    print(f'       Augmented vector shape: {X_aug.shape}  ✓')

    # ── Extract labels ────────────────────────────────────────
    label_col = 'Finding Labels' if 'Finding Labels' in df.columns else df.columns[-1]
    print(f'       Using label column    : "{label_col}"')

    # Handle both pipe-separated string and pre-binarized columns
    if df[label_col].dtype == object:
        labels = df[label_col].fillna('No Finding').apply(
            lambda x: [p.strip() for p in str(x).split('|')]
        ).tolist()
    else:
        labels = df[label_col].tolist()

    return X_aug, labels, df


# ── Load both splits ──────────────────────────────────────────
X_train_aug, train_labels_raw, df_train = load_and_augment(
    TRAIN_FEATURES_PATH, TRAIN_CSV_PATH, 'Train'
)
X_test_aug, test_labels_raw, df_test = load_and_augment(
    TEST_FEATURES_PATH, TEST_CSV_PATH, 'Test'
)

print(f'\\n[DATA] Final Augmented Shapes:')
print(f'       X_train_aug : {X_train_aug.shape}')
print(f'       X_test_aug  : {X_test_aug.shape}')

In [ ]:
def binarize_labels(train_labels_raw, test_labels_raw, pathologies):
    """
    Fits a MultiLabelBinarizer on the 14 defined pathologies and
    transforms both train and test label sets.
    """
    mlb = MultiLabelBinarizer(classes=pathologies)

    y_train = mlb.fit_transform(train_labels_raw)
    y_test  = mlb.transform(test_labels_raw)

    print(f'[MLB] MultiLabelBinarizer fitted on {len(pathologies)} classes')
    print(f'      y_train shape : {y_train.shape}  dtype={y_train.dtype}')
    print(f'      y_test  shape : {y_test.shape}  dtype={y_test.dtype}')
    print(f'      Classes       : {list(mlb.classes_)}')

    # Label distribution
    print('\\n[MLB] Training Label Distribution:')
    for i, cls in enumerate(mlb.classes_):
        pos = y_train[:, i].sum()
        pct = 100.0 * pos / len(y_train)
        print(f'      [{i+1:2d}] {cls:<22s}  positives={pos:6d}  ({pct:.2f}%)')

    return mlb, y_train.astype(np.float32), y_test.astype(np.float32)


mlb, y_train, y_test = binarize_labels(train_labels_raw, test_labels_raw, PATHOLOGIES)

In [ ]:
class MetaContextualMLP(nn.Module):
    """
    4-Layer MLP for the Neural Stream of the MCE.

    Architecture:
        Input  (1027)
          → Linear(1027, 1024) → BatchNorm → ReLU → Dropout(0.5)
          → Linear(1024,  512) → BatchNorm → ReLU → Dropout(0.5)
          → Linear( 512,  256) → BatchNorm → ReLU → Dropout(0.5)
          → Linear( 256,   14) → Sigmoid
    """
    def __init__(self, input_dim=1027, num_classes=14, dropout=0.5):
        super(MetaContextualMLP, self).__init__()

        self.network = nn.Sequential(
            # ── Layer 1: Input → 1024 ─────────────────────
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),

            # ── Layer 2: 1024 → 512 ───────────────────────
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),

            # ── Layer 3: 512 → 256 ────────────────────────
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),

            # ── Layer 4 (Output): 256 → 14 ────────────────
            nn.Linear(256, num_classes),
            nn.Sigmoid()
        )

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.network(x)


# ── Instantiate model ─────────────────────────────────────────
mlp_model = MetaContextualMLP(
    input_dim=CONFIG['AUGMENTED_DIM'],
    num_classes=NUM_PATHOLOGIES,
    dropout=CONFIG['DROPOUT_RATE']
).to(DEVICE)

print('[MLP] MetaContextualMLP Architecture:')
print(mlp_model)

# Parameter count
total_params = sum(p.numel() for p in mlp_model.parameters())
trainable_params = sum(p.numel() for p in mlp_model.parameters() if p.requires_grad)
print(f'\\n[MLP] Total Parameters     : {total_params:,}')
print(f'[MLP] Trainable Parameters : {trainable_params:,}')

In [ ]:
class WeightedBinaryCrossEntropyLoss(nn.Module):
    """
    Custom Weighted Binary Cross-Entropy Loss for clinical calibration.

    For each pathology c:
        loss_c = -[ w_c * y * log(p) + (1 - y) * log(1 - p) ]

    where w_c is the positivity weight from the clinical prior.
    Higher weights penalize missed detections of rarer/critical pathologies.
    """
    def __init__(self, pos_weights: torch.Tensor, reduction='mean'):
        """
        Args:
            pos_weights : Tensor of shape (num_classes,) — per-class positive weights
            reduction   : 'mean' | 'sum' | 'none'
        """
        super(WeightedBinaryCrossEntropyLoss, self).__init__()
        self.register_buffer('pos_weights', pos_weights)
        self.reduction = reduction

        print('[LOSS] WeightedBCE initialized with positivity weights:')
        for i, (p, w) in enumerate(zip(PATHOLOGIES, pos_weights.cpu().tolist())):
            print(f'       [{i+1:2d}] {p:<22s}  w={w:.2f}')

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        """
        Args:
            y_pred : Tensor (batch, num_classes) — sigmoid probabilities [0,1]
            y_true : Tensor (batch, num_classes) — binary ground truth {0,1}
        Returns:
            Scalar loss
        """
        eps = 1e-7  # numerical stability
        y_pred = torch.clamp(y_pred, eps, 1.0 - eps)

        # Weighted positive term + standard negative term
        loss = -(
            self.pos_weights * y_true * torch.log(y_pred) +
            (1.0 - y_true) * torch.log(1.0 - y_pred)
        )  # shape: (batch, num_classes)

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


criterion = WeightedBinaryCrossEntropyLoss(pos_weights=POS_WEIGHTS).to(DEVICE)

# ── Optimizer & Scheduler ─────────────────────────────────────
optimizer = optim.AdamW(
    mlp_model.parameters(),
    lr=CONFIG['LEARNING_RATE'],
    weight_decay=1e-4
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True
)
print('\\n[OPTIM] AdamW optimizer + ReduceLROnPlateau scheduler ready.')

In [ ]:
from sklearn.model_selection import train_test_split

# ── Train / Validation split for neural stream ────────────────
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_aug, y_train,
    test_size=CONFIG['META_TEST_SIZE'],
    random_state=42
)

def make_dataloader(X, y, batch_size, shuffle=True):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.float32)
    ds  = TensorDataset(X_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=(DEVICE.type == 'cuda'))

train_loader = make_dataloader(X_tr, y_tr,    CONFIG['BATCH_SIZE'], shuffle=True)
val_loader   = make_dataloader(X_val, y_val,  CONFIG['BATCH_SIZE'], shuffle=False)
test_loader  = make_dataloader(X_test_aug, y_test, CONFIG['BATCH_SIZE'], shuffle=False)

print(f'[DATA] Train loader  : {len(train_loader)} batches  ({len(X_tr)} samples)')
print(f'[DATA] Val   loader  : {len(val_loader)} batches  ({len(X_val)} samples)')
print(f'[DATA] Test  loader  : {len(test_loader)} batches  ({len(X_test_aug)} samples)')

In [ ]:
def train_neural_stream(model, train_loader, val_loader, criterion, optimizer,
                         scheduler, epochs, patience, device, save_path):
    """
    Full training loop with early stopping and best-model checkpointing.
    """
    history = {'train_loss': [], 'val_loss': [], 'val_auc': []}
    best_val_loss = float('inf')
    patience_counter = 0
    best_epoch = 0

    print(f'\\n[TRAIN] Starting Neural Stream training for {epochs} epochs...')
    print('='*70)

    for epoch in range(1, epochs + 1):
        start_time = time.time()

        # ── Training phase ────────────────────────────────
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            preds = model(X_batch)
            loss  = criterion(preds, y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
        train_loss /= len(train_loader.dataset)

        # ── Validation phase ──────────────────────────────
        model.eval()
        val_loss  = 0.0
        all_preds = []
        all_true  = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                preds  = model(X_batch)
                loss   = criterion(preds, y_batch)
                val_loss += loss.item() * X_batch.size(0)
                all_preds.append(preds.cpu().numpy())
                all_true.append(y_batch.cpu().numpy())
        val_loss /= len(val_loader.dataset)

        all_preds = np.concatenate(all_preds, axis=0)
        all_true  = np.concatenate(all_true,  axis=0)

        # Per-class AUC (skip classes with no positives in val)
        val_aucs = []
        for c in range(NUM_PATHOLOGIES):
            if all_true[:, c].sum() > 0:
                try:
                    val_aucs.append(roc_auc_score(all_true[:, c], all_preds[:, c]))
                except Exception:
                    pass
        mean_auc = np.mean(val_aucs) if val_aucs else 0.0

        scheduler.step(val_loss)
        elapsed = time.time() - start_time

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_auc'].append(mean_auc)

        print(f'  Epoch [{epoch:3d}/{epochs}] '
              f'Train Loss: {train_loss:.4f} | '
              f'Val Loss: {val_loss:.4f} | '
              f'Val AUC: {mean_auc:.4f} | '
              f'Time: {elapsed:.1f}s')

        # ── Early Stopping & Checkpointing ────────────────
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
            print(f'           ✓ Best model saved (val_loss={best_val_loss:.4f})')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'\\n[TRAIN] Early stopping at epoch {epoch} '
                      f'(best epoch={best_epoch})')
                break

    print('='*70)
    print(f'[TRAIN] Training complete. Best epoch: {best_epoch}, '
          f'Best val loss: {best_val_loss:.4f}')
    return history


MLP_SAVE_PATH = os.path.join(MODEL_SAVE_DIR, 'mlp_neural_stream_best.pt')

history = train_neural_stream(
    model       = mlp_model,
    train_loader= train_loader,
    val_loader  = val_loader,
    criterion   = criterion,
    optimizer   = optimizer,
    scheduler   = scheduler,
    epochs      = CONFIG['EPOCHS'],
    patience    = CONFIG['EARLY_STOP_PATIENCE'],
    device      = DEVICE,
    save_path   = MLP_SAVE_PATH
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Neural Stream (MLP) Training History', fontsize=14, fontweight='bold')

epochs_ran = range(1, len(history['train_loss']) + 1)

# Loss curves
axes[0].plot(epochs_ran, history['train_loss'], 'b-o', markersize=3, label='Train Loss')
axes[0].plot(epochs_ran, history['val_loss'],   'r-o', markersize=3, label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Weighted BCE Loss')
axes[0].set_title('Loss Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# AUC curve
axes[1].plot(epochs_ran, history['val_auc'], 'g-o', markersize=3, label='Val Mean AUC')
axes[1].axhline(y=0.8, color='orange', linestyle='--', label='AUC=0.80 threshold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Mean AUC-ROC')
axes[1].set_title('Validation Mean AUC-ROC')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 1])

plt.tight_layout()
plot_path = os.path.join(MODEL_SAVE_DIR, 'training_curves.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'[PLOT] Training curves saved → {plot_path}')

In [ ]:
print('[SVM] Training Statistical Stream (SVM + RBF) ...')
print(f'      OneVsRest over {NUM_PATHOLOGIES} pathologies')
print(f'      C={CONFIG["SVM_C"]}, kernel={CONFIG["SVM_KERNEL"]}')
print('      Note: Using CalibratedClassifierCV for probability output')

# SVM with RBF kernel, wrapped for probability calibration
base_svm = SVC(
    kernel=CONFIG['SVM_KERNEL'],
    C=CONFIG['SVM_C'],
    probability=True,     # Enable probability estimates via Platt scaling
    random_state=42,
    class_weight='balanced',
    cache_size=500
)

svm_classifier = OneVsRestClassifier(
    base_svm,
    n_jobs=-1
)

start = time.time()
svm_classifier.fit(X_train_aug, y_train.astype(int))
elapsed = time.time() - start

print(f'[SVM] Training complete in {elapsed:.1f}s')

# ── Get probability outputs ───────────────────────────────────
svm_train_probs = svm_classifier.predict_proba(X_train_aug)
svm_test_probs  = svm_classifier.predict_proba(X_test_aug)
print(f'[SVM] Train probs shape : {svm_train_probs.shape}')
print(f'[SVM] Test  probs shape : {svm_test_probs.shape}')

# ── Quick AUC check ───────────────────────────────────────────
svm_aucs = []
for c in range(NUM_PATHOLOGIES):
    if y_test[:, c].sum() > 0:
        auc = roc_auc_score(y_test[:, c], svm_test_probs[:, c])
        svm_aucs.append(auc)
        print(f'      {PATHOLOGIES[c]:<22s}  AUC={auc:.4f}')
print(f'\\n[SVM] Mean AUC (test): {np.mean(svm_aucs):.4f}')

# Save model
joblib.dump(svm_classifier, os.path.join(MODEL_SAVE_DIR, 'svm_statistical_stream.pkl'))
print('[SVM] Model saved.')

In [ ]:
print('[RF] Training Non-Linear Stream (Random Forest) ...')
print(f'     n_estimators={CONFIG["RF_N_ESTIMATORS"]}, OneVsRest')

base_rf = RandomForestClassifier(
    n_estimators=CONFIG['RF_N_ESTIMATORS'],
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
    oob_score=True
)

rf_classifier = OneVsRestClassifier(base_rf, n_jobs=-1)

start = time.time()
rf_classifier.fit(X_train_aug, y_train.astype(int))
elapsed = time.time() - start
print(f'[RF] Training complete in {elapsed:.1f}s')

# ── Get probability outputs ───────────────────────────────────
rf_train_probs = rf_classifier.predict_proba(X_train_aug)
rf_test_probs  = rf_classifier.predict_proba(X_test_aug)
print(f'[RF] Train probs shape : {rf_train_probs.shape}')
print(f'[RF] Test  probs shape : {rf_test_probs.shape}')

# ── Quick AUC check ───────────────────────────────────────────
rf_aucs = []
for c in range(NUM_PATHOLOGIES):
    if y_test[:, c].sum() > 0:
        auc = roc_auc_score(y_test[:, c], rf_test_probs[:, c])
        rf_aucs.append(auc)
        print(f'     {PATHOLOGIES[c]:<22s}  AUC={auc:.4f}')
print(f'\\n[RF] Mean AUC (test): {np.mean(rf_aucs):.4f}')

# Save model
joblib.dump(rf_classifier, os.path.join(MODEL_SAVE_DIR, 'rf_nonlinear_stream.pkl'))
print('[RF] Model saved.')

In [ ]:
def get_mlp_probabilities(model, dataloader, device):
    """Run inference on a DataLoader and return probability array."""
    model.eval()
    all_probs = []
    with torch.no_grad():
        for X_batch, _ in dataloader:
            X_batch = X_batch.to(device)
            probs   = model(X_batch)
            all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs, axis=0)

# Load best checkpoint
mlp_model.load_state_dict(torch.load(MLP_SAVE_PATH, map_location=DEVICE))
print('[MLP] Best checkpoint loaded.')

# Full train probabilities (for meta-learner training)
train_mlp_loader = make_dataloader(X_train_aug, y_train, CONFIG['BATCH_SIZE'], shuffle=False)
mlp_train_probs  = get_mlp_probabilities(mlp_model, train_mlp_loader, DEVICE)
mlp_test_probs   = get_mlp_probabilities(mlp_model, test_loader, DEVICE)

print(f'[MLP] Train probs shape : {mlp_train_probs.shape}')
print(f'[MLP] Test  probs shape : {mlp_test_probs.shape}')

# ── Quick AUC check ───────────────────────────────────────────
mlp_aucs = []
for c in range(NUM_PATHOLOGIES):
    if y_test[:, c].sum() > 0:
        auc = roc_auc_score(y_test[:, c], mlp_test_probs[:, c])
        mlp_aucs.append(auc)
        print(f'  {PATHOLOGIES[c]:<22s}  AUC={auc:.4f}')
print(f'\\n[MLP] Mean AUC (test): {np.mean(mlp_aucs):.4f}')

In [ ]:
print('[MCE] Building Meta-Contextual Ensemble (MCE) ...')
print('      Stacking: SVM_probs + RF_probs + MLP_probs → Meta-Learner')

# ── Concatenate all stream outputs ───────────────────────────
# Shape: (N, 14*3) = (N, 42)  →  meta-features for 14 pathologies
meta_train = np.concatenate([
    svm_train_probs,  # (N, 14)
    rf_train_probs,   # (N, 14)
    mlp_train_probs   # (N, 14)
], axis=1).astype(np.float32)

meta_test = np.concatenate([
    svm_test_probs,   # (N, 14)
    rf_test_probs,    # (N, 14)
    mlp_test_probs    # (N, 14)
], axis=1).astype(np.float32)

print(f'[MCE] Meta-train features shape : {meta_train.shape}')
print(f'[MCE] Meta-test  features shape : {meta_test.shape}')

# ── Per-class Logistic Regression meta-learner ────────────────
meta_learners = []
mce_test_probs = np.zeros((len(X_test_aug), NUM_PATHOLOGIES), dtype=np.float32)

print('\\n[MCE] Training per-pathology meta-learners...')
for c in range(NUM_PATHOLOGIES):
    # Features for class c: [SVM_c, RF_c, MLP_c] = 3 values per sample
    # Actually we pass all 42 features for cross-class context
    meta_lr = LogisticRegression(
        C=1.0,
        max_iter=1000,
        random_state=42,
        class_weight='balanced',
        solver='lbfgs'
    )
    meta_lr.fit(meta_train, y_train[:, c].astype(int))
    mce_test_probs[:, c] = meta_lr.predict_proba(meta_test)[:, 1]
    meta_learners.append(meta_lr)
    print(f'  [{c+1:2d}] {PATHOLOGIES[c]:<22s}  meta-LR fitted ✓')

# Save meta-learners
joblib.dump(meta_learners, os.path.join(MODEL_SAVE_DIR, 'mce_meta_learners.pkl'))
print('\\n[MCE] All meta-learners saved.')

In [ ]:
def evaluate_ensemble(y_true, y_probs, pathologies, threshold=0.5, stream_name='MCE'):
    """
    Full evaluation: per-class AUC-ROC, Average Precision, F1,
    and macro averages.
    """
    print(f'\\n{"="*70}')
    print(f'  EVALUATION REPORT — {stream_name}')
    print(f'{"="*70}')

    y_pred = (y_probs >= threshold).astype(int)

    results = []
    aucs, aps, f1s = [], [], []

    print(f'  {"Pathology":<22s}  {"AUC-ROC":>8s}  {"Avg Prec":>9s}  '
          f'{"F1":>6s}  {"Support":>8s}')
    print(f'  {"-"*22}  {"-"*8}  {"-"*9}  {"-"*6}  {"-"*8}')

    for c, name in enumerate(pathologies):
        support = int(y_true[:, c].sum())
        if support == 0:
            auc, ap, f1 = 0.0, 0.0, 0.0
        else:
            auc = roc_auc_score(y_true[:, c], y_probs[:, c])
            ap  = average_precision_score(y_true[:, c], y_probs[:, c])
            f1  = f1_score(y_true[:, c], y_pred[:, c], zero_division=0)
            aucs.append(auc)
            aps.append(ap)
            f1s.append(f1)

        results.append({'Pathology': name, 'AUC-ROC': auc,
                        'Avg Precision': ap, 'F1': f1, 'Support': support})
        print(f'  {name:<22s}  {auc:>8.4f}  {ap:>9.4f}  {f1:>6.4f}  {support:>8d}')

    print(f'  {"-"*22}  {"-"*8}  {"-"*9}  {"-"*6}  {"-"*8}')
    print(f'  {"MACRO AVERAGE":<22s}  {np.mean(aucs):>8.4f}  '
          f'{np.mean(aps):>9.4f}  {np.mean(f1s):>6.4f}')
    print(f'\\n  Hamming Loss  : {hamming_loss(y_true, y_pred):.6f}')
    print(f'  Threshold     : {threshold}')
    print('='*70)

    return pd.DataFrame(results)


# ── Evaluate all streams + MCE ────────────────────────────────
df_svm_results = evaluate_ensemble(y_test, svm_test_probs,  PATHOLOGIES, stream_name='Statistical Stream (SVM)')
df_rf_results  = evaluate_ensemble(y_test, rf_test_probs,   PATHOLOGIES, stream_name='Non-Linear Stream (RF)')
df_mlp_results = evaluate_ensemble(y_test, mlp_test_probs,  PATHOLOGIES, stream_name='Neural Stream (MLP)')
df_mce_results = evaluate_ensemble(y_test, mce_test_probs,  PATHOLOGIES, stream_name='MCE FINAL ENSEMBLE')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

x     = np.arange(NUM_PATHOLOGIES)
width = 0.2

bars_svm = ax.bar(x - 1.5*width, df_svm_results['AUC-ROC'], width,
                  label='SVM (Statistical)', color='#4C72B0', alpha=0.85)
bars_rf  = ax.bar(x - 0.5*width, df_rf_results['AUC-ROC'],  width,
                  label='RF (Non-Linear)',   color='#55A868', alpha=0.85)
bars_mlp = ax.bar(x + 0.5*width, df_mlp_results['AUC-ROC'], width,
                  label='MLP (Neural)',      color='#C44E52', alpha=0.85)
bars_mce = ax.bar(x + 1.5*width, df_mce_results['AUC-ROC'], width,
                  label='MCE Ensemble',      color='#8172B2', alpha=0.95)

ax.axhline(y=0.80, color='orange', linestyle='--', linewidth=1.5,
           label='Clinical Threshold (AUC=0.80)')
ax.set_xlabel('Pathology', fontsize=12)
ax.set_ylabel('AUC-ROC Score', fontsize=12)
ax.set_title('MCE vs Individual Streams: Per-Pathology AUC-ROC\\n'
             'Multi-Label Chest X-Ray Diagnostic System', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(PATHOLOGIES, rotation=45, ha='right', fontsize=9)
ax.set_ylim([0, 1.05])
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
auc_plot_path = os.path.join(MODEL_SAVE_DIR, 'comparative_auc_roc.png')
plt.savefig(auc_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'[PLOT] Comparative AUC chart saved → {auc_plot_path}')

In [ ]:
auc_matrix = np.array([
    df_svm_results['AUC-ROC'].values,
    df_rf_results['AUC-ROC'].values,
    df_mlp_results['AUC-ROC'].values,
    df_mce_results['AUC-ROC'].values,
])

stream_names = ['SVM (Statistical)', 'RF (Non-Linear)', 'MLP (Neural)', 'MCE Ensemble']

fig, ax = plt.subplots(figsize=(16, 4))
sns.heatmap(
    auc_matrix,
    annot=True, fmt='.3f', cmap='RdYlGn',
    xticklabels=PATHOLOGIES,
    yticklabels=stream_names,
    vmin=0.5, vmax=1.0,
    linewidths=0.5,
    ax=ax
)
ax.set_title('AUC-ROC Heatmap: All Streams vs 14 Pathologies',
             fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
heatmap_path = os.path.join(MODEL_SAVE_DIR, 'auc_heatmap.png')
plt.savefig(heatmap_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'[PLOT] AUC heatmap saved → {heatmap_path}')

In [ ]:
summary = pd.DataFrame({
    'Pathology'           : PATHOLOGIES,
    'SVM_AUC'             : df_svm_results['AUC-ROC'].values,
    'RF_AUC'              : df_rf_results['AUC-ROC'].values,
    'MLP_AUC'             : df_mlp_results['AUC-ROC'].values,
    'MCE_AUC'             : df_mce_results['AUC-ROC'].values,
    'MCE_AvgPrecision'    : df_mce_results['Avg Precision'].values,
    'MCE_F1'              : df_mce_results['F1'].values,
    'Support'             : df_mce_results['Support'].values,
    'PosWeight'           : POS_WEIGHTS.cpu().numpy(),
})

summary_path = os.path.join(MODEL_SAVE_DIR, 'phase4_mce_results.csv')
summary.to_csv(summary_path, index=False)
print(f'[SAVE] Results CSV saved → {summary_path}')
print()
print(summary.to_string(index=False))

# ── Final Summary ─────────────────────────────────────────────
print('\\n' + '='*70)
print('  PHASE 4 – MCE FINAL SUMMARY')
print('='*70)
print(f'  Architecture     : Meta-Contextual Ensemble (MCE)')
print(f'  Input Dimension  : {CONFIG["AUGMENTED_DIM"]} (1024 visual + 3 clinical)')
print(f'  Pathologies      : {NUM_PATHOLOGIES}')
print(f'  SVM  Mean AUC    : {df_svm_results["AUC-ROC"].mean():.4f}')
print(f'  RF   Mean AUC    : {df_rf_results["AUC-ROC"].mean():.4f}')
print(f'  MLP  Mean AUC    : {df_mlp_results["AUC-ROC"].mean():.4f}')
print(f'  MCE  Mean AUC    : {df_mce_results["AUC-ROC"].mean():.4f}  ← FINAL')
print(f'  MCE  Mean F1     : {df_mce_results["F1"].mean():.4f}')
print(f'  MCE  Mean AvgP   : {df_mce_results["Avg Precision"].mean():.4f}')
print('='*70)
print(f'\\n  Models saved in : {MODEL_SAVE_DIR}')
print('  Files:')
for fname in os.listdir(MODEL_SAVE_DIR):
    fpath = os.path.join(MODEL_SAVE_DIR, fname)
    size  = os.path.getsize(fpath) / 1024
    print(f'    {fname:<45s}  {size:>8.1f} KB')